# NN: bigram

In [1]:
words = open('names.txt').read().splitlines()

In [2]:
import torch
import torch.nn.functional as F


In [3]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0

itos = {i:s for s, i in stoi.items()}

In [4]:
print('chars:', len(chars))

chars: 26


In [38]:
# create the dataset
xs, ys = [], []

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        xs.append(stoi[ch1])
        ys.append(stoi[ch2])

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()

print('number of examples:', num)

number of examples: 228146


In [39]:
ys.shape

torch.Size([228146])

In [7]:
xenc = F.one_hot(xs, num_classes=27).float()
xenc

tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 1., 0.],
        [0., 0., 0.,  ..., 0., 0., 1.],
        [0., 0., 0.,  ..., 1., 0., 0.]])

In [43]:
g = torch.Generator()
W = torch.randn((27, 27), generator=g, requires_grad=True)
num_classes = len(chars) + 1

for k in range(300):
    # forward pass
    xenc = F.one_hot(xs, num_classes=num_classes).float()
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)
    print("probs =", probs.shape)

    loss = -probs[torch.arange(num), ys].log().mean() + 0.01 * (W ** 2).mean()
    print(loss.item())

    # backward pass
    W.grad = None # set to zero the gradient
    loss.backward()

    # update
    W.data += -50 * W.grad


probs = torch.Size([228146, 27])
3.8834173679351807
probs = torch.Size([228146, 27])
3.4945857524871826
probs = torch.Size([228146, 27])
3.230903387069702
probs = torch.Size([228146, 27])
3.052072286605835
probs = torch.Size([228146, 27])
2.9369587898254395
probs = torch.Size([228146, 27])
2.853976011276245
probs = torch.Size([228146, 27])
2.7907862663269043
probs = torch.Size([228146, 27])
2.7428019046783447
probs = torch.Size([228146, 27])
2.706493377685547
probs = torch.Size([228146, 27])
2.6788389682769775
probs = torch.Size([228146, 27])
2.657400131225586
probs = torch.Size([228146, 27])
2.640352487564087
probs = torch.Size([228146, 27])
2.626432418823242
probs = torch.Size([228146, 27])
2.614800214767456
probs = torch.Size([228146, 27])
2.6048998832702637
probs = torch.Size([228146, 27])
2.5963475704193115
probs = torch.Size([228146, 27])
2.5888688564300537
probs = torch.Size([228146, 27])
2.5822606086730957
probs = torch.Size([228146, 27])
2.5763673782348633
probs = torch.Size([

In [9]:
for i in range(10):
    ix = 0
    out = []
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        logits = xenc @ W
        counts = logits.exp()
        p = counts / counts.sum(1, keepdim=True)

        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[int(ix)])
        if ix == 0:
            break

    print(''.join(out))

la.
m.
kamenebridhahmas.
rezari.
hiolknrs.
beryalonandyn.
lio.
rmelycin.
aneiauelli.
trkarinayneviefrienawialahinahies.


# E01

train a trigram language model, i.e. take two characters as an input to predict the 3rd one. Feel free to use either counting or a neural net. Evaluate the loss; Did it improve over a bigram model?

In [50]:
# create the dataset
xs, ys = [], []

for w in words:
    chs = ['.', '.'] + list(w) + ['.']
    # print("chs =", chs)
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        xs.append([stoi[ch1], stoi[ch2]])
        ys.append(stoi[ch3])

# print("xs:", xs)
# print("ys:", ys)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.shape[0]

print('number of examples:', num)

number of examples: 228146


In [46]:
xs.shape

torch.Size([5, 2])

In [47]:
xs[0]

tensor([0, 0])

In [48]:
F.one_hot(xs, num_classes=num_classes).float().shape

torch.Size([5, 2, 27])

In [53]:
g = torch.Generator()
W = torch.randn((54, 27), generator=g, requires_grad=True)
num_classes = len(chars) + 1

for k in range(300):
    # forward pass
    xenc = F.one_hot(xs, num_classes=num_classes).float()
    xenc = xenc.reshape(-1, 54)
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)
    # print("probs =", probs.shape)
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01 * (W ** 2).mean()
    print(loss.item())

    # backward pass
    W.grad = None # set to zero the gradient
    loss.backward()

    # update
    W.data += -30 * W.grad

4.182948589324951
3.675206184387207
3.347933769226074
3.1671228408813477
3.0441393852233887
2.9555530548095703
2.887425661087036
2.8328795433044434
2.7881953716278076
2.7509663105010986
2.719494104385376
2.6925318241119385
2.669158697128296
2.648688793182373
2.6306087970733643
2.6145267486572266
2.60013747215271
2.5871970653533936
2.5755064487457275
2.5649008750915527
2.5552423000335693
2.546412944793701
2.538313150405884
2.5308568477630615
2.5239715576171875
2.5175929069519043
2.5116665363311768
2.5061445236206055
2.5009868144989014
2.4961564540863037
2.491621971130371
2.487356185913086
2.4833357334136963
2.4795379638671875
2.475944757461548
2.4725399017333984
2.4693069458007812
2.4662344455718994
2.4633092880249023
2.4605209827423096
2.457860231399536
2.455317497253418
2.4528849124908447
2.450556516647339
2.448323965072632
2.4461822509765625
2.4441254138946533
2.4421489238739014
2.4402475357055664
2.4384171962738037
2.4366540908813477
2.4349541664123535
2.433314323425293
2.4317309856

In [58]:
for i in range(10):
    ix = [0, 0]
    out = []
    while True:
        xenc = F.one_hot(torch.tensor(ix), num_classes=27).float()
        xenc = xenc.reshape(-1, 54)

        logits = xenc @ W
        counts = logits.exp()
        p = counts / counts.sum(1, keepdim=True)

        ix_new = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        ix.append(ix_new)
        ix.pop(0)
        out.append(itos[int(ix_new)])

        if ix_new == 0:
            break

    print(''.join(out))

myshantabis.
darisalvonn.
sitselynn.
ziad.
maellimaridyawili.
do.
kan.
rucyron.
beran.
kishniveoxion.


In [61]:
xs, ys = [], []

for w in words:
    chs = ['.', '.', '.'] + list(w) + ['.']
    # print("chs =", chs)
    for ch1, ch2, ch3, ch4 in zip(chs, chs[1:], chs[2:], chs[3:]):
        xs.append([stoi[ch1], stoi[ch2], stoi[ch3]])
        ys.append(stoi[ch4])

# print("xs:", xs)
# print("ys:", ys)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.shape[0]

In [65]:
g = torch.Generator()
W = torch.randn((81, 27), generator=g, requires_grad=True)
num_classes = len(chars) + 1

for k in range(1000):
    # forward pass
    xenc = F.one_hot(xs, num_classes=num_classes).float()
    xenc = xenc.reshape(-1, 81)
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)
    # print("probs =", probs.shape)
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01 * (W ** 2).mean()
    print(loss.item())

    # backward pass
    W.grad = None # set to zero the gradient
    loss.backward()

    # update
    W.data += -5 * W.grad

4.448252201080322
4.297523498535156
4.163247585296631
4.0407257080078125
3.9298925399780273
3.832289695739746
3.7492258548736572
3.6800646781921387
3.6221301555633545
3.572183847427368
3.527681827545166
3.487065315246582
3.4494788646698
3.4144508838653564
3.381683588027954
3.3509538173675537
3.322075605392456
3.2948834896087646
3.269227981567383
3.244971990585327
3.2219934463500977
3.200181007385254
3.179435968399048
3.159670114517212
3.1408040523529053
3.122767448425293
3.105498790740967
3.088941812515259
3.073047637939453
3.0577712059020996
3.04307222366333
3.028916358947754
3.0152697563171387
3.002103090286255
2.989389657974243
2.977104663848877
2.9652254581451416
2.9537320137023926
2.942603826522827
2.931824207305908
2.9213757514953613
2.9112439155578613
2.901414155960083
2.891871690750122
2.882606029510498
2.873603582382202
2.864853858947754
2.856346607208252
2.848071813583374
2.840019702911377
2.8321826457977295
2.8245503902435303
2.8171164989471436
2.809873104095459
2.8028128147

In [66]:
for i in range(10):
    ix = [0, 0, 0]
    out = []
    while True:
        xenc = F.one_hot(torch.tensor(ix), num_classes=27).float()
        xenc = xenc.reshape(-1, 81)

        logits = xenc @ W
        counts = logits.exp()
        p = counts / counts.sum(1, keepdim=True)

        ix_new = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        ix.append(ix_new)
        ix.pop(0)
        out.append(itos[int(ix_new)])

        if ix_new == 0:
            break

    print(''.join(out))

alydolinra.
beryano.
falyn.
lan.
ruelye.
melyea.
bellin.
rka.
kelanevie.
biena.


# Test Dataset
E02: split up the dataset randomly into 80% train set, 10% dev set, 10% test set. Train the bigram and trigram models only on the training set. Evaluate them on dev and test splits. What can you see?

In [87]:
# create the dataset
import random

def build_dataset(words):
    xs, ys = [], []

    for w in words:
        chs = ['.'] + list(w) + ['.']
        for ch1, ch2 in zip(chs, chs[1:]):
            xs.append(stoi[ch1])
            ys.append(stoi[ch2])

    return torch.tensor(xs), torch.tensor(ys)

random.shuffle(words)
n1 = int(len(words) * 0.8)
n2 = int(len(words) * 0.9)

train_words = words[:n1]
dev_words = words[n1:n2]
test_words = words[n2:]

xs_train, ys_train = build_dataset(train_words)
xs_dev, ys_dev = build_dataset(dev_words)
xs_test, ys_test = build_dataset(test_words)

In [88]:
# train
g = torch.Generator()
W = torch.randn((27, 27), generator=g, requires_grad=True)
num_classes = len(chars) + 1

for k in range(100):
    # forward pass
    xenc = F.one_hot(xs_train, num_classes=num_classes).float()
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)
    print("probs =", probs.shape)

    loss = -probs[torch.arange(len(xs_train)), ys_train].log().mean() + 0.01 * (W ** 2).mean()
    print(loss.item())

    # backward pass
    W.grad = None # set to zero the gradient
    loss.backward()

    # update
    W.data += -50 * W.grad


probs = torch.Size([182509, 27])
3.88321852684021
probs = torch.Size([182509, 27])
3.4945871829986572
probs = torch.Size([182509, 27])
3.230645179748535
probs = torch.Size([182509, 27])
3.0516138076782227
probs = torch.Size([182509, 27])
2.9365079402923584
probs = torch.Size([182509, 27])
2.8534915447235107
probs = torch.Size([182509, 27])
2.7903454303741455
probs = torch.Size([182509, 27])
2.742424726486206
probs = torch.Size([182509, 27])
2.7061665058135986
probs = torch.Size([182509, 27])
2.6785385608673096
probs = torch.Size([182509, 27])
2.65710186958313
probs = torch.Size([182509, 27])
2.6400370597839355
probs = torch.Size([182509, 27])
2.626084089279175
probs = torch.Size([182509, 27])
2.614412546157837
probs = torch.Size([182509, 27])
2.6044704914093018
probs = torch.Size([182509, 27])
2.5958807468414307
probs = torch.Size([182509, 27])
2.5883708000183105
probs = torch.Size([182509, 27])
2.581737518310547
probs = torch.Size([182509, 27])
2.5758261680603027
probs = torch.Size([1

In [89]:
def evaluate(x, y, w):
    with torch.no_grad():
        xenc = F.one_hot(x, num_classes=num_classes).float()
        logits = xenc @ w
        counts = logits.exp()
        probs = counts / counts.sum(1, keepdim=True)
        return -probs[torch.arange(len(x)), y].log().mean() + 0.01 * (w ** 2).mean()

In [90]:
print("train loss =", evaluate(xs_train, ys_train, W))
print("dev loss =", evaluate(xs_dev, ys_dev, W))
print("test loss =", evaluate(xs_test, ys_test, W))

train loss = tensor(2.4898)
dev loss = tensor(2.4984)
test loss = tensor(2.4887)
